# Workflow :
* Step 1: Data Acquisition & Cleaning
    - Ingestion, Pruning, Data Normalization
* Step 2: Sequential Feature Engineering
    - Item Explosion, Temporal Bucketing, Cart State Tracking
* Step 3: AI-Driven Semantic Categorization
    - Semantic Embedding, Clustering, Cold Start Mitigation
* Step 4: Model Development & Training
    - Algorithm Selection(LightGBM), Training Strategy, Optimization
* Step 5: Real-Time Inference & Business Impact
    - Offline Evaluation, Impact Simulation, Insights


### Step 1: Data Acquisition & Cleaning


In [ ]:
import pandas as pd
import re

In [ ]:
df = pd.read_csv('Zomato data.csv', encoding = 'latin-1')

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
for col in df.columns:
    print(col)

In [ ]:
cols =[
'KPT duration (minutes)',
'Rider wait time (minutes)',
'Order Ready Marked',
'Customer complaint tag',
'Cancellation / Rejection reason',
'Restaurant compensation (Cancellation)',
'Restaurant penalty (Rejection)',
'Instructions',
'Review',
'Packaging charges']

df.drop(cols, axis=1, inplace=True)

In [ ]:
df.head()
# df.info()

In [ ]:
df['Discount construct'] = df['Discount construct'].fillna('No Discount')

In [ ]:
df['Rating_missing'] = df['Rating'].isna().astype(int)
df['Rating'] = df['Rating'].fillna(df['Rating'].mean())

In [ ]:
df.info()

In [ ]:
df['Order Placed At'] = pd.to_datetime(df['Order Placed At'], format='%I:%M %p, %B %d %Y')

df['order_hour'] = df['Order Placed At'].dt.hour
df['day_of_week'] = df['Order Placed At'].dt.day_name()
df['is_weekend'] = df['Order Placed At'].dt.weekday.isin([5,6]).astype(int)

In [ ]:
#type casting of distance
df['Distance'] = df['Distance'].str.replace('km', '', regex=False)
df['Distance'] = df['Distance'].str.replace('<1', '0.5', regex=False)
df['Distance'] = df['Distance'].str.strip()
df['Distance'] = pd.to_numeric(df['Distance'], errors='coerce')
print(df['Distance'].isnull().sum())

In [ ]:
print(df['Order Status'].unique())
df = df[df['Order Status'] == 'Delivered']

In [ ]:
print(df['Bill subtotal'].describe())
print(df['Total'].describe())

In [ ]:
df['Total_Discount'] = (
    df['Restaurant discount (Promo)'] +
    df['Restaurant discount (Flat offs, Freebies & others)'] +
    df['Gold discount'] +
    df['Brand pack discount']
)

In [ ]:
print(df['Items in order'].iloc[0])

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
for col in df.columns:
    print(f"{col:<30} Missing: {df[col].isnull().sum():<5} Type: {df[col].dtype}")

In [ ]:
df.head(2)

In [ ]:
df.head(2)

### Step 2: Sequential Feature Engineering


In [ ]:
# 1. Explode items
items_df = df[['Order ID', 'Customer ID', 'Restaurant ID', 'Items in order']].copy()
items_df['Items List'] = items_df['Items in order'].str.split(', ')
items_df = items_df.explode('Items List')
items_df.rename(columns={'Items List': 'Item Name'}, inplace=True)
items_df['Quantity'] = items_df['Item Name'].str.extract(r'(\d+)').astype(int)
items_df['Item Name'] = items_df['Item Name'].str.replace(r'^\d+\s*x\s*', '', regex=True)

# 2. Merge back other order-level columns
cols_to_keep = [c for c in df.columns if c not in ['Items in order', 'Order ID', 'Customer ID', 'Restaurant ID']]
item_level_df = items_df.merge(df[cols_to_keep + ['Order ID','Customer ID','Restaurant ID']], 
                               on=['Order ID','Customer ID','Restaurant ID'],
                               how='left')

# 3. Reorder columns
item_level_df = item_level_df[['Order ID','Customer ID','Restaurant ID','Restaurant name','City','Subzone',
                               'Order Placed At','order_hour','day_of_week','is_weekend','Delivery','Distance',
                               'Item Name','Quantity','Bill subtotal','Total','Total_Discount',
                               'Restaurant discount (Promo)','Restaurant discount (Flat offs, Freebies & others)',
                               'Gold discount','Brand pack discount','Rating','Rating_missing']]

# 4. Check
pd.set_option('display.max_columns', None)
item_level_df.head(5)

In [ ]:
# Explode items into multiple rows in main df
df = df.copy()  # optional
df['Items List'] = df['Items in order'].str.split(', ')
df = df.explode('Items List').reset_index(drop=True)

# Extract quantity and item name
df['Quantity'] = df['Items List'].str.extract(r'(\d+)').astype(int)
df['Item Name'] = df['Items List'].str.replace(r'^\d+\s*x\s*', '', regex=True)

# Drop the old column if needed
df.drop(columns=['Items in order', 'Items List'], inplace=True)

df.head()

In [ ]:
df.info()

In [ ]:

for col in df.columns:
    print(f"{col:<30} Missing: {df[col].isnull().sum():<5} Type: {df[col].dtype}")

In [ ]:
df.head()

In [ ]:
print("Unique Users:", df["Customer ID"].nunique())
print("Unique Restaurants:", df["Restaurant ID"].nunique())
print("Unique Items:", df["Item Name"].nunique())
print("Average items per order:", df.groupby("Order ID")["Item Name"].count().mean())

In [ ]:
order_sizes = df.groupby("Order ID")["Item Name"].count()

print("Total Orders:", order_sizes.shape[0])
print("Orders with 2+ items:", (order_sizes >= 2).sum())
print("Orders with 3+ items:", (order_sizes >= 3).sum())

In [ ]:


# Make sure your exploded dataset is called df
# Columns: Customer ID, Order ID, Item Name, Order Placed At, Order Hour, Day of Week, Is Weekend, etc.

# First, sort the items within each order by Order Placed At
df_sorted = df.sort_values(['Order ID', 'Order Placed At'])

# Create a function to generate sequential samples
def create_sequential_samples(df):
    samples = []

    # Group by each order
    for order_id, group in df.groupby('Order ID'):
        items = list(group['Item Name'].values) # type: ignore
        customer_id = group['Customer ID'].iloc[0]
        restaurant_id = group['Restaurant ID'].iloc[0]
        order_hour = group['order_hour'].iloc[0]
        day_of_week = group['day_of_week'].iloc[0]
        is_weekend = group['is_weekend'].iloc[0]

        # Only consider orders with 2+ items
        if len(items) < 2:
            continue

        # Create sequential cart samples
        for i in range(1, len(items)):
            cart = list(items[:i])    
            target = items[i]        

            samples.append({
                'Customer ID': customer_id,
                'Restaurant ID': restaurant_id,
                'Cart Items': cart,
                'Target Item': target,
                'Order Hour': order_hour,
                'Day of Week': day_of_week,
                'Is Weekend': is_weekend
            })

    return pd.DataFrame(samples)

# Generate training samples
train_df = create_sequential_samples(df_sorted)

print("Number of training samples:", train_df.shape[0])
train_df.head(10)



In [ ]:
train_df['cart_size'] = train_df['Cart Items'].apply(len)

In [ ]:
beverages = ['Coke', 'Pepsi', 'Juice', 'Cold Drink']

def has_beverage(cart):
    return int(any(b in item for item in cart for b in beverages))

train_df['has_beverage'] = train_df['Cart Items'].apply(has_beverage)

In [ ]:
desserts = ['Cake', 'Gulab', 'Ice Cream', 'Brownie']

def has_dessert(cart):
    return int(any(d in item for item in cart for d in desserts))

train_df['has_dessert'] = train_df['Cart Items'].apply(has_dessert)

In [ ]:
def meal_bucket(hour):
    if 6 <= hour < 10:
        return 'breakfast'
    elif 11 <= hour < 16:
        return 'lunch'
    elif 18 <= hour < 22:
        return 'dinner'
    else:
        return 'late_night'

train_df['meal_bucket'] = train_df['Order Hour'].apply(meal_bucket)

In [ ]:
rest_pop = df.groupby('Restaurant ID')['Item Name'].count().to_dict()
train_df['restaurant_popularity'] = train_df['Restaurant ID'].map(rest_pop)

In [ ]:
user_avg_items = df.groupby('Customer ID')['Item Name'].count() / df.groupby('Customer ID')['Order ID'].nunique()
train_df['user_avg_items'] = train_df['Customer ID'].map(user_avg_items)

In [ ]:
item_pop = df['Item Name'].value_counts().to_dict()
train_df['last_item_in_cart'] = train_df['Cart Items'].apply(lambda c: c[-1])
train_df['last_item_popularity'] = train_df['last_item_in_cart'].map(item_pop)

from sklearn.preprocessing import LabelEncoder
le_items = LabelEncoder()
# Fit on all unique items so we can encode last_item_in_cart
all_possible_items = list(item_pop.keys())
le_items.fit(all_possible_items)
train_df['last_item_encoded'] = le_items.transform(train_df['last_item_in_cart'])

In [ ]:
meal_mapping = {'breakfast':0, 'lunch':1, 'dinner':2, 'late_night':3}
train_df['meal_bucket_encoded'] = train_df['meal_bucket'].map(meal_mapping)

### Step 3: AI-Driven Semantic Categorization



In [ ]:
# Flatten all items from Cart Items + Target Item
all_cart_items = [item for sublist in train_df['Cart Items'] for item in sublist]
all_target_items = train_df['Target Item'].tolist()

unique_items = list(set(all_cart_items + all_target_items))
print("Total unique items:", len(unique_items))
print("Some examples:", unique_items[:10])

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load embedding model
model_embed = SentenceTransformer('all-MiniLM-L6-v2')

# Convert items to embeddings
item_embeddings = model_embed.encode(unique_items, convert_to_numpy=True)

In [ ]:
# Embeddings logic

In [ ]:
# Seed items for each category
main_items = ["Pizza", "Burger", "Chicken", "Biryani", "Fries", "Paneer", "Pasta"]
beverages = ["Coke", "Pepsi", "Sprite", "Maza", "Juice", "Coffee", "Tea"]
desserts = ["Gulab Jamun", "Ice Cream", "Cake", "Rasmalai", "Barfi", "Brownie"]

# Convert seed items to embeddings
main_embed = model_embed.encode(main_items, convert_to_numpy=True)
bev_embed = model_embed.encode(beverages, convert_to_numpy=True)
dessert_embed = model_embed.encode(desserts, convert_to_numpy=True)

# Step 1: Categorize all unique items
def categorize_item(item, item_emb):
    sim_main = cosine_similarity([item_emb], main_embed).max()
    sim_bev = cosine_similarity([item_emb], bev_embed).max()
    sim_dessert = cosine_similarity([item_emb], dessert_embed).max()
    
    sims = {"main": sim_main, "beverage": sim_bev, "dessert": sim_dessert}
    return max(list(sims.keys()), key=lambda k: sims[k])

item_category_mapping = {}
for item, emb in zip(unique_items, item_embeddings):
    item_category_mapping[item] = categorize_item(item, emb)

# Quick check
for i, (k, v) in enumerate(item_category_mapping.items()):
    if i >= 10: break
    print(k, "->", v)

# Step 2: Add AI-based features to the dataset
# Cart-level features: does the cart have a beverage or dessert?
def cart_has_category(cart_items, category_name):
    return int(any(item_category_mapping[item] == category_name for item in cart_items))

train_df['has_beverage'] = train_df['Cart Items'].apply(lambda x: cart_has_category(x, "beverage"))
train_df['has_dessert'] = train_df['Cart Items'].apply(lambda x: cart_has_category(x, "dessert"))
train_df['has_main'] = train_df['Cart Items'].apply(lambda x: cart_has_category(x, "main"))



###  Step 4: Model Development & Training


In [ ]:
from sklearn.model_selection import train_test_split
import lightgbm as lgb
import numpy as np

# Features
feature_cols = [
    'cart_size',
    'has_beverage',
    'has_dessert',
    'has_main',
    'Order Hour',
    'meal_bucket_encoded',
    'restaurant_popularity',
    'user_avg_items',
    'last_item_popularity',
    'last_item_encoded'
]

X = train_df[feature_cols]
y = train_df['Target Item']

# Encode target items
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)
inverse_target_mapping = {i: str(item) for i, item in enumerate(le.classes_)}

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42
)
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


### Step 5: Real-Time Inference & Business Impact


In [ ]:

import time
from sklearn.metrics import roc_auc_score, ndcg_score

print("\n--- Training Model ---")
start_time = time.time()
model = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=len(le.classes_),
    learning_rate=0.05,
    n_estimators=200,
    num_leaves=127,
    max_depth=10,
    verbose=-1,
    random_state=42
)
model.fit(X_train, y_train)
train_time = time.time() - start_time
print(f"Model trained in {train_time:.2f} seconds")

# Predict
print("\n--- Generating Predictions ---")
start_time = time.time()
y_pred_probs = model.predict_proba(X_test)
inference_time = time.time() - start_time
print(f"Inference time for {len(X_test)} samples: {inference_time:.3f} seconds ({(inference_time / len(X_test)) * 1000:.2f} ms per request)")

print("\n--- Offline Evaluation Metrics ---")

def precision_recall_at_k(y_true, y_pred_probs, classes, k=10):
    top_k_indices = np.argsort(y_pred_probs, axis=1)[:, -k:][:, ::-1]
    top_k_preds = classes[top_k_indices]
    hits = 0
    for i in range(len(y_true)):
        top_items_list = list(top_k_preds[i])
        if y_true[i] in top_items_list:
            hits += 1
    N = len(y_true)
    precision = hits / (N * k)
    recall = hits / N
    return precision, recall

p_at_k, r_at_k = precision_recall_at_k(y_test, y_pred_probs, model.classes_, k=10)
print(f"Precision@10: {p_at_k:.4f}")
print(f"Recall@10: {r_at_k:.4f}")

try:
    y_test_onehot = np.zeros_like(y_pred_probs)
    for i, y_t in enumerate(y_test):
        idx = np.where(model.classes_ == y_t)[0]
        if len(idx) > 0:
            y_test_onehot[i, idx[0]] = 1
    ndcg_k = ndcg_score(y_test_onehot, y_pred_probs, k=10)
    print(f"NDCG@10: {ndcg_k:.4f}")
except Exception as e:
    print(f"Could not compute NDCG: {e}")

# Proxies
accepted_pct = 0.10 # Assume 10% acceptance out of people shown the top K additions
item_prices = df.groupby('Item Name')['Total'].mean().to_dict() # approximate item price via average transaction size
valid_prices = [item_prices.get(le.classes_[i], 0) for i in y_test]
avg_add_on_price = np.mean(valid_prices)
aov_lift = accepted_pct * avg_add_on_price

print(f"\n--- Estimated Business Impact ---")
print(f"Assume 10% Acceptance Rate (A/B Test proxy)")
print(f"Average expected AOV Lift per targeted cart: INR {aov_lift:.2f}")

### Visualisation

In [ ]:

import os
import matplotlib.pyplot as plt
import seaborn as sns

print("\n--- Generating Visualizations ---")
os.makedirs('plots', exist_ok=True)

# 1. Feature Importance
plt.figure(figsize=(10, 6))
importance = model.feature_importances_
sns.barplot(x=importance, y=feature_cols)
plt.title('Feature Importance in CSAO LightGBM Model')
plt.xlabel('Importance (Split)')
plt.ylabel('Features')
plt.tight_layout()
plt.savefig('plots/feature_importance.png')
plt.show()
plt.close()

# 2. Top 10 Recommendations Distribution
recommended_categories = []
for probs in y_pred_probs:
    top_items_idx = np.argsort(probs)[::-1][:10]
    for idx in top_items_idx:
        item_name = inverse_target_mapping.get(idx, "")
        cat = item_category_mapping.get(item_name, 'Other')
        recommended_categories.append(cat)

plt.figure(figsize=(8, 5))
sns.countplot(x=recommended_categories, order=['main', 'beverage', 'dessert', 'Other'])
plt.title('Distribution of Categories in Top 10 Recommendations')
plt.xlabel('Category')
plt.ylabel('Frequency in Top 10')
plt.tight_layout()
plt.savefig('plots/recommendation_categories.png')
plt.show()
plt.close()

print("Visualizations saved to 'plots/' directory.")
